In [ ]:
import numpy as np
import pandas as pd
from numpy.char import lower
from datetime import timedelta

File_Path = "D:\\Work\\Climate Adaption Planner\\SmartFarming\\Data_Setup\\Datasets\\crop_recommendationV3.csv" 


In [ ]:
def read_crop_data(file_path):
    """
    Reads crop data from a CSV file and returns it as a pandas DataFrame.

    Parameters:
    file_path (str): The path to the CSV file containing crop data.

    Returns:
    pd.DataFrame: A DataFrame containing the crop data.
    """
    try:
        crop_data = pd.read_csv(file_path)
        return crop_data
    except Exception as e:
        print(f"An error occurred while reading the crop data: {e}")
        return None

In [ ]:
crop_data = read_crop_data(File_Path)

## Exploratory Data Analysis

Before extracting rules, we profile the dataset across three lenses:
1. **Categorical distributions** — how are crop, season, growth stage, soil type distributed?
2. **Numeric feature distributions per crop** — do crops occupy distinct feature ranges?
3. **Correlation structure** — which features move together, and which are independent?

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ── Theme constants ────────────────────────────────────────────────────────────
BG      = '#0f1117'
AX_BG   = '#1a1d27'
WHITE   = '#f1f5f9'
GREY    = '#6b7280'
AMBER   = '#fbbf24'
GREEN   = '#4ade80'
BLUE    = '#60a5fa'
RED     = '#f87171'
PURPLE  = '#a78bfa'

CROP_COLORS = {'cotton': BLUE, 'maize': AMBER, 'rice': GREEN}
CROPS       = sorted(crop_data['label'].unique())

def style_ax(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(AX_BG)
    for sp in ax.spines.values(): sp.set_edgecolor('#2d3148')
    ax.tick_params(colors=WHITE, labelsize=8)
    if title:   ax.set_title(title,   color=WHITE, fontsize=9,  fontweight='bold', pad=7)
    if xlabel:  ax.set_xlabel(xlabel, color=WHITE, fontsize=8)
    if ylabel:  ax.set_ylabel(ylabel, color=WHITE, fontsize=8)

# ── Figure 1: Categorical distributions ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.patch.set_facecolor(BG)
fig.suptitle('Dataset Overview — Categorical & Structural Distributions',
             color=WHITE, fontsize=12, fontweight='bold', y=0.98)

# 1a. Crop counts
ax = axes[0, 0]
counts = crop_data['label'].value_counts()
bars = ax.bar(counts.index, counts.values,
              color=[CROP_COLORS[c] for c in counts.index], alpha=0.85)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', color=WHITE, fontsize=9)
style_ax(ax, 'Crop Distribution', ylabel='Count')

# 1b. Season per crop (stacked bar)
ax = axes[0, 1]
season_ct = crop_data.groupby(['label', 'season']).size().unstack(fill_value=0)
seasons   = season_ct.columns.tolist()
s_colors  = [PURPLE, BLUE, GREEN]
bottom    = [0] * len(season_ct)
for i, (season, col) in enumerate(zip(seasons, s_colors)):
    vals = season_ct[season].values
    bars = ax.bar(season_ct.index, vals, bottom=bottom, color=col, alpha=0.85,
                  label=season)
    bottom = [b + v for b, v in zip(bottom, vals)]
ax.legend(fontsize=7, labelcolor=WHITE, facecolor=AX_BG, edgecolor=GREY)
style_ax(ax, 'Season Distribution per Crop', ylabel='Count')

# 1c. Growth stage counts
ax = axes[0, 2]
gs_map  = {1: 'Seedling', 2: 'Growing', 3: 'Mature'}
gs_ct   = crop_data['growth_stage'].value_counts().sort_index()
gs_lbls = [gs_map[k] for k in gs_ct.index]
bars    = ax.bar(gs_lbls, gs_ct.values, color=[BLUE, GREEN, AMBER], alpha=0.85)
for bar, val in zip(bars, gs_ct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', color=WHITE, fontsize=9)
style_ax(ax, 'Growth Stage Distribution', ylabel='Count')

# 1d. Soil type counts
ax = axes[1, 0]
st_map = {1: 'Type 1', 2: 'Type 2', 3: 'Type 3'}
st_ct  = crop_data['soil_type'].value_counts().sort_index()
bars   = ax.bar([st_map[k] for k in st_ct.index], st_ct.values,
                color=[BLUE, GREEN, AMBER], alpha=0.85)
for bar, val in zip(bars, st_ct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', color=WHITE, fontsize=9)
style_ax(ax, 'Soil Type Distribution', ylabel='Count')

# 1e. Water source type
ax = axes[1, 1]
ws_ct = crop_data['water_source_type'].value_counts().sort_index()
bars  = ax.bar([f'Source {k}' for k in ws_ct.index], ws_ct.values,
               color=[BLUE, GREEN, AMBER], alpha=0.85)
for bar, val in zip(bars, ws_ct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', color=WHITE, fontsize=9)
style_ax(ax, 'Water Source Distribution', ylabel='Count')

# 1f. Crop × stage × soil combination counts (heatmap)
ax = axes[1, 2]
combo = crop_data.groupby(['label', 'growth_stage', 'soil_type']).size().reset_index(name='n')
combo['x_label'] = combo['growth_stage'].map(gs_map) + '\n' + 'Soil ' + combo['soil_type'].astype(str)
pivot_combo = combo.pivot_table(index='label', columns=['growth_stage', 'soil_type'], values='n', fill_value=0)
im = ax.imshow(pivot_combo.values, cmap='YlGn', aspect='auto')
ax.set_yticks(range(len(pivot_combo.index))); ax.set_yticklabels(pivot_combo.index, fontsize=8)
ax.set_xticks(range(pivot_combo.shape[1]))
xlabels = [f'S{gs}\nSoil{sl}' for gs, sl in pivot_combo.columns]
ax.set_xticklabels(xlabels, fontsize=6.5)
for r in range(pivot_combo.shape[0]):
    for c in range(pivot_combo.shape[1]):
        ax.text(c, r, str(pivot_combo.values[r, c]), ha='center', va='center',
                color='#111', fontsize=8, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(colors=WHITE, labelsize=7)
style_ax(ax, 'Samples per Crop × Stage × Soil')

for ax in axes.flat:
    ax.set_facecolor(AX_BG)
    for sp in ax.spines.values(): sp.set_edgecolor('#2d3148')
    ax.tick_params(colors=WHITE)

plt.tight_layout()
# plt.savefig('eda_categorical.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ── Figure 2: Numeric feature distributions per crop ────────────
ALL_RULE_COLS = [
    'temperature', 'humidity', 'sunlight_exposure', 'wind_speed',
    'co2_concentration', 'crop_density', 'frost_risk',
    'N', 'P', 'K', 'ph', 'organic_matter', 'soil_moisture',
    'rainfall', 'irrigation_frequency',
]

n_cols = 5
n_rows = -(-len(ALL_RULE_COLS) // n_cols)   # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.2))
fig.patch.set_facecolor(BG)
fig.suptitle('Rulebook Feature Distributions by Crop', color=WHITE,
             fontsize=12, fontweight='bold', y=1.01)

for idx, col in enumerate(ALL_RULE_COLS):
    ax = axes[idx // n_cols][idx % n_cols]
    data_by_crop = [crop_data.loc[crop_data['label'] == c, col].values for c in CROPS]
    vp = ax.violinplot(data_by_crop, positions=range(len(CROPS)),
                       showmedians=True, showextrema=True)
    for i, (body, crop) in enumerate(zip(vp['bodies'], CROPS)):
        body.set_facecolor(CROP_COLORS[crop])
        body.set_alpha(0.6)
    vp['cmedians'].set_color(WHITE)
    vp['cbars'].set_color(GREY)
    vp['cmaxes'].set_color(GREY)
    vp['cmins'].set_color(GREY)
    ax.set_xticks(range(len(CROPS)))
    ax.set_xticklabels(CROPS, fontsize=7.5)
    style_ax(ax, col)

# Hide unused axes
for idx in range(len(ALL_RULE_COLS), n_rows * n_cols):
    axes[idx // n_cols][idx % n_cols].set_visible(False)

plt.tight_layout()
# plt.savefig('eda_feature_distributions.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ── Figure 3: Correlation heatmap ─────────────────────────────────────────────
import numpy as np

corr = crop_data[ALL_RULE_COLS].corr()

fig, ax = plt.subplots(figsize=(13, 11))
fig.patch.set_facecolor(BG)
ax.set_facecolor(AX_BG)

# Diverging colormap centred at 0
cmap = plt.cm.RdYlGn
im   = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')

n = len(ALL_RULE_COLS)
ax.set_xticks(range(n)); ax.set_xticklabels(ALL_RULE_COLS, rotation=45, ha='right',
                                              fontsize=8.5, color=WHITE)
ax.set_yticks(range(n)); ax.set_yticklabels(ALL_RULE_COLS, fontsize=8.5, color=WHITE)

for i in range(n):
    for j in range(n):
        val  = corr.values[i, j]
        text_col = '#111' if 0.3 < abs(val) < 0.8 else WHITE
        if i != j:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=7, color=text_col)

cb = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cb.ax.tick_params(colors=WHITE, labelsize=8)
cb.set_label('Pearson r', color=WHITE, fontsize=9)

for sp in ax.spines.values(): sp.set_edgecolor('#2d3148')
ax.tick_params(colors=WHITE)
ax.set_title('Feature Correlation Matrix — Rulebook Columns',
             color=WHITE, fontsize=12, fontweight='bold', pad=12)

plt.tight_layout()
# plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## Range-Based RuleBooks

Extracts `(min, max, mean, std)` for every numeric column,  
grouped by **crop (`label`) × `growth_stage` × `soil_type`**.

Output: `climate_rulebook[crop][growth_stage][soil_type][column]`

In [ ]:
import json

# ── Key columns ────────────────────────────────────────────────────────────────
CROP_STAGE_SOIL = ['label', 'growth_stage', 'soil_type']
CROP_ONLY       = ['label']

# ── Column sets per rulebook ───────────────────────────────────────────────────
# Float-only continuous measurements (excludes categorically-encoded ints:
#   season, water_source_type, irrigation_frequency, soil_type, growth_stage)
# Also excluded entirely: pest_pressure, fertilizer_usage, urban_area_proximity

CLIMATE_COLS   = ['temperature', 'humidity', 'sunlight_exposure',
                  'wind_speed', 'co2_concentration', 'crop_density', 'frost_risk']

SOIL_CHEM_COLS = ['N', 'P', 'K', 'ph', 'organic_matter', 'soil_moisture']
# N/P/K are integer-typed but continuous nutrient quantities (22–77 unique values)

WATER_COLS     = ['soil_moisture', 'rainfall']

# ── Quick sanity check ────────────────────────────────────────────────────────
print(f'Dataset      : {crop_data.shape[0]} rows, {crop_data.shape[1]} columns')
print(f'Crops        : {sorted(crop_data["label"].unique())}')
print(f'Growth stages: {sorted(crop_data["growth_stage"].unique())}')
print(f'Soil types   : {sorted(crop_data["soil_type"].unique())}')
print()
print(f'climate_rulebook   cols ({len(CLIMATE_COLS)})  : {CLIMATE_COLS}')
print(f'soil_chemistry cols ({len(SOIL_CHEM_COLS)})  : {SOIL_CHEM_COLS}')
print(f'water_rulebook cols ({len(WATER_COLS)})  : {WATER_COLS}')

In [ ]:
def extract_ranges(df, key_cols, range_cols):
    """
    Group df by key_cols; compute min/max/mean/std for each column in range_cols.
    Returns nested dict keyed by the group values, leaf = {col: {min,max,mean,std}}.
    """
    rulebook = {}
    for keys, group in df.groupby(key_cols):
        # Normalise single-key groups to a tuple for uniform handling
        keys = (keys,) if not isinstance(keys, tuple) else keys
        col_ranges = {}
        for col in range_cols:
            col_ranges[col] = {
                'min' : round(float(group[col].min()),  4),
                'max' : round(float(group[col].max()),  4),
                'mean': round(float(group[col].mean()), 4),
                'std' : round(float(group[col].std()),  4),
            }
        # Build nested dict from the key tuple
        node = rulebook
        for k in keys[:-1]:
            node = node.setdefault(k if isinstance(k, str) else int(k), {})
        node[keys[-1] if isinstance(keys[-1], str) else int(keys[-1])] = col_ranges
    return rulebook


# ── Build the three rulebooks ─────────────────────────────────────────────────
climate_rulebook       = extract_ranges(crop_data, CROP_STAGE_SOIL, CLIMATE_COLS)
soil_chemistry_rulebook = extract_ranges(crop_data, CROP_ONLY,       SOIL_CHEM_COLS)
water_rulebook         = extract_ranges(crop_data, CROP_STAGE_SOIL, WATER_COLS)

# ── Summary ───────────────────────────────────────────────────────────────────
def _count_leaves(d, depth):
    if depth == 1:
        return len(d)
    return sum(_count_leaves(v, depth - 1) for v in d.values())

print(f'climate_rulebook        : {_count_leaves(climate_rulebook,       3)} combinations × {len(CLIMATE_COLS)} cols')
print(f'soil_chemistry_rulebook : {_count_leaves(soil_chemistry_rulebook, 1)} combinations × {len(SOIL_CHEM_COLS)} cols')
print(f'water_rulebook          : {_count_leaves(water_rulebook,          3)} combinations × {len(WATER_COLS)} cols')

In [ ]:
def print_rulebook_sample(title, rulebook, *keys):
    """Pretty-print the rules for one set of keys."""
    node = rulebook
    for k in keys:
        node = node[k]
    label = ' | '.join(str(k) for k in keys)
    print(f'{title}  [{label}]\n')
    header = f'{"Column":<22} {"Min":>10} {"Max":>10} {"Mean":>10} {"Std":>10}'
    print(header)
    print('-' * len(header))
    for col, s in node.items():
        print(f'{col:<22} {s["min"]:>10} {s["max"]:>10} {s["mean"]:>10} {s["std"]:>10}')
    print()


print_rulebook_sample('climate_rulebook',        climate_rulebook,        'rice', 1, 2)
print_rulebook_sample('soil_chemistry_rulebook', soil_chemistry_rulebook, 'rice')
print_rulebook_sample('water_rulebook',          water_rulebook,          'rice', 1, 2)

In [ ]:
def rulebook_to_df(rulebook, key_names):
    """Flatten any depth of nested rulebook into a long-format DataFrame."""
    records = []

    def _walk(node, keys_so_far):
        if isinstance(node, dict):
            # Leaf level: values are {min,max,mean,std} dicts
            first_val = next(iter(node.values()))
            if isinstance(first_val, dict) and 'min' in first_val:
                for col, stats in node.items():
                    records.append({**dict(zip(key_names, keys_so_far)), 'column': col, **stats})
            else:
                for k, v in node.items():
                    _walk(v, keys_so_far + [k])

    _walk(rulebook, [])
    return pd.DataFrame(records)


climate_df    = rulebook_to_df(climate_rulebook,        ['crop', 'growth_stage', 'soil_type'])
soil_chem_df  = rulebook_to_df(soil_chemistry_rulebook, ['crop'])
water_df      = rulebook_to_df(water_rulebook,          ['crop', 'growth_stage', 'soil_type'])

print(f'climate_df     : {climate_df.shape}')
print(f'soil_chem_df   : {soil_chem_df.shape}')
print(f'water_df       : {water_df.shape}')
print()
soil_chem_df  # soil_chemistry has the simplest shape — good quick check

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CLIMATE RULEBOOK — Visualization
# Key: crop × growth_stage × soil_type
# Cols: temperature, humidity, sunlight_exposure, wind_speed,
#       co2_concentration, crop_density, frost_risk
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor(BG)
gs_fig = gridspec.GridSpec(3, 3, figure=fig, hspace=0.52, wspace=0.38)
fig.suptitle('Climate Rulebook — Extracted Ranges (crop × stage × soil)',
             color=WHITE, fontsize=12, fontweight='bold', y=0.99)

gs_map = {1: 'Seedling', 2: 'Growing', 3: 'Mature'}

# ── Panel 1-7: Mean heatmaps — one per climate column (crop × stage, soil avg) ─
for idx, col in enumerate(CLIMATE_COLS):
    row, col_pos = divmod(idx, 3)
    ax = fig.add_subplot(gs_fig[row, col_pos])
    pivot = crop_data.groupby(['label', 'growth_stage'])[col].mean().unstack()
    pivot.columns = [gs_map[c] for c in pivot.columns]
    im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(pivot.shape[1])); ax.set_xticklabels(pivot.columns, fontsize=8)
    ax.set_yticks(range(pivot.shape[0])); ax.set_yticklabels(pivot.index, fontsize=8)
    for r in range(pivot.shape[0]):
        for c in range(pivot.shape[1]):
            ax.text(c, r, f'{pivot.values[r, c]:.1f}', ha='center', va='center',
                    color='#111', fontsize=8.5, fontweight='bold')
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(colors=WHITE, labelsize=6)
    style_ax(ax, f'Mean {col}  (crop × stage, soil avg)')

# Hide last panel — use it for radar chart
ax_radar = fig.add_subplot(gs_fig[2, 2], polar=True)
ax_radar.set_facecolor(AX_BG)

# ── Panel 9: Radar chart — average climate profile per crop ────────────────────
from matplotlib.patches import FancyArrowPatch

# Normalise each feature to [0,1] for the radar
climate_means = crop_data.groupby('label')[CLIMATE_COLS].mean()
norm_min = climate_means.min()
norm_max = climate_means.max()
norm_vals = (climate_means - norm_min) / (norm_max - norm_min + 1e-9)

angles = np.linspace(0, 2 * np.pi, len(CLIMATE_COLS), endpoint=False).tolist()
angles += angles[:1]   # close the polygon

ax_radar.set_facecolor(AX_BG)
ax_radar.spines['polar'].set_color('#2d3148')
ax_radar.set_theta_offset(np.pi / 2)
ax_radar.set_theta_direction(-1)
ax_radar.set_rlabel_position(30)
ax_radar.tick_params(colors=WHITE, labelsize=6.5)
ax_radar.yaxis.set_tick_params(labelcolor=GREY)
ax_radar.set_ylim(0, 1)

short_labels = ['Temp', 'Humid', 'Sun', 'Wind', 'CO₂', 'Density', 'Frost']
ax_radar.set_thetagrids(np.degrees(angles[:-1]), short_labels,
                        fontsize=8, color=WHITE)

for crop in CROPS:
    vals = norm_vals.loc[crop].tolist() + [norm_vals.loc[crop].tolist()[0]]
    ax_radar.plot(angles, vals, color=CROP_COLORS[crop], linewidth=2, label=crop)
    ax_radar.fill(angles, vals, color=CROP_COLORS[crop], alpha=0.12)

ax_radar.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15),
                fontsize=8, labelcolor=WHITE, facecolor=AX_BG, edgecolor=GREY)
ax_radar.set_title('Climate Profile per Crop\n(normalised means)',
                   color=WHITE, fontsize=9, fontweight='bold', pad=18)

# plt.savefig('climate_rulebook_viz.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SOIL CHEMISTRY RULEBOOK — Visualization
# Key: crop only
# Cols: N, P, K, ph, organic_matter, soil_moisture
# ══════════════════════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor(BG)
gs_fig = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.38)
fig.suptitle('Soil Chemistry Rulebook — Ranges per Crop (crop-level key)',
             color=WHITE, fontsize=12, fontweight='bold', y=0.99)

# ── Panels 1–3: Range interval plots — [min, mean±std, max] per crop ──────────
for idx, col in enumerate(SOIL_CHEM_COLS[:3]):   # N, P, K
    ax = fig.add_subplot(gs_fig[0, idx])
    for i, crop in enumerate(CROPS):
        grp  = crop_data.loc[crop_data['label'] == crop, col]
        mn, mx, mu, sd = grp.min(), grp.max(), grp.mean(), grp.std()
        c = CROP_COLORS[crop]
        # Full range bar
        ax.plot([mn, mx], [i, i], color=c, linewidth=3, alpha=0.4, solid_capstyle='round')
        # Mean ± std band
        ax.barh(i, sd * 2, left=mu - sd, height=0.35, color=c, alpha=0.7)
        # Mean dot
        ax.plot(mu, i, 'o', color=WHITE, markersize=6, zorder=5)
        ax.text(mx + (mx - mn) * 0.04, i, f'{mu:.1f}', va='center',
                color=WHITE, fontsize=8)
    ax.set_yticks(range(len(CROPS))); ax.set_yticklabels(CROPS, fontsize=9)
    ax.axvline(0, color=GREY, linewidth=0.5)
    style_ax(ax, f'{col} — Range per Crop\n(bar=mean±std, line=min–max, dot=mean)',
             xlabel=col)

# ── Panels 4–6: ph, organic_matter, soil_moisture ─────────────────────────────
for idx, col in enumerate(SOIL_CHEM_COLS[3:]):
    ax = fig.add_subplot(gs_fig[1, idx])
    for i, crop in enumerate(CROPS):
        grp  = crop_data.loc[crop_data['label'] == crop, col]
        mn, mx, mu, sd = grp.min(), grp.max(), grp.mean(), grp.std()
        c = CROP_COLORS[crop]
        ax.plot([mn, mx], [i, i], color=c, linewidth=3, alpha=0.4, solid_capstyle='round')
        ax.barh(i, sd * 2, left=mu - sd, height=0.35, color=c, alpha=0.7)
        ax.plot(mu, i, 'o', color=WHITE, markersize=6, zorder=5)
        ax.text(mx + (mx - mn) * 0.04, i, f'{mu:.2f}', va='center',
                color=WHITE, fontsize=8)
    ax.set_yticks(range(len(CROPS))); ax.set_yticklabels(CROPS, fontsize=9)
    style_ax(ax, f'{col} — Range per Crop\n(bar=mean±std, line=min–max, dot=mean)',
             xlabel=col)

# ── Radar: soil chemistry profiles per crop (normalised) ──────────────────────
# Replace the last subplot with a polar axes
fig.get_axes()[-1].remove()
ax_r = fig.add_subplot(gs_fig[1, 2], polar=True)

soil_means = crop_data.groupby('label')[SOIL_CHEM_COLS].mean()
s_min = soil_means.min(); s_max = soil_means.max()
s_norm = (soil_means - s_min) / (s_max - s_min + 1e-9)

s_angles = np.linspace(0, 2 * np.pi, len(SOIL_CHEM_COLS), endpoint=False).tolist()
s_angles += s_angles[:1]

ax_r.set_facecolor(AX_BG)
ax_r.spines['polar'].set_color('#2d3148')
ax_r.set_theta_offset(np.pi / 2); ax_r.set_theta_direction(-1)
ax_r.set_ylim(0, 1); ax_r.set_rlabel_position(30)
ax_r.tick_params(colors=WHITE, labelsize=6.5)
ax_r.yaxis.set_tick_params(labelcolor=GREY)
ax_r.set_thetagrids(np.degrees(s_angles[:-1]),
                    ['N', 'P', 'K', 'pH', 'Org.Matter', 'Soil Moist.'],
                    fontsize=8, color=WHITE)

for crop in CROPS:
    vals = s_norm.loc[crop].tolist() + [s_norm.loc[crop].tolist()[0]]
    ax_r.plot(s_angles, vals, color=CROP_COLORS[crop], linewidth=2.5, label=crop)
    ax_r.fill(s_angles, vals, color=CROP_COLORS[crop], alpha=0.15)

ax_r.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15),
            fontsize=8, labelcolor=WHITE, facecolor=AX_BG, edgecolor=GREY)
ax_r.set_title('Soil Chemistry Profile\n(normalised crop means)',
               color=WHITE, fontsize=9, fontweight='bold', pad=18)

# plt.savefig('soil_chemistry_rulebook_viz.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WATER RULEBOOK — Visualization
# Key: crop × growth_stage × soil_type
# Cols: soil_moisture, rainfall
# ══════════════════════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor(BG)
gs_fig = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.40)
fig.suptitle('Water Rulebook — Ranges per Crop × Stage × Soil',
             color=WHITE, fontsize=12, fontweight='bold', y=0.99)

gs_map = {1: 'Seedling', 2: 'Growing', 3: 'Mature'}

# ── Helper: make a group label for x-axis ─────────────────────────────────────
def group_label(crop, stage, soil):
    return f'{crop[:3]}\nS{stage}-Soil{soil}'

groups = [(c, s, t)
          for c in sorted(crop_data['label'].unique())
          for s in sorted(crop_data['growth_stage'].unique())
          for t in sorted(crop_data['soil_type'].unique())]
xlabels = [group_label(*g) for g in groups]

for col_idx, col in enumerate(WATER_COLS):
    # ── Row 0: Interval range plot (min–max, mean dot) across all 27 groups ──
    ax = fig.add_subplot(gs_fig[0, col_idx])
    for i, (crop, stage, soil) in enumerate(groups):
        grp = crop_data.loc[
            (crop_data['label'] == crop) &
            (crop_data['growth_stage'] == stage) &
            (crop_data['soil_type'] == soil), col]
        mn, mx, mu, sd = grp.min(), grp.max(), grp.mean(), grp.std()
        c = CROP_COLORS[crop]
        ax.plot([mn, mx], [i, i], color=c, linewidth=2.5, alpha=0.35, solid_capstyle='round')
        ax.barh(i, sd * 2, left=mu - sd, height=0.55, color=c, alpha=0.65)
        ax.plot(mu, i, 'o', color=WHITE, markersize=4.5, zorder=5)
    ax.set_yticks(range(len(groups)))
    ax.set_yticklabels(xlabels, fontsize=6.5)
    patches = [mpatches.Patch(color=CROP_COLORS[c], label=c) for c in CROPS]
    ax.legend(handles=patches, fontsize=7, labelcolor=WHITE,
              facecolor=AX_BG, edgecolor=GREY, loc='lower right')
    style_ax(ax, f'{col} — Range across all groups\n(bar=mean±std, line=min–max)',
             xlabel=col)

    # ── Row 1: Heatmap — mean value (crop × stage, soil avg) ─────────────────
    ax2 = fig.add_subplot(gs_fig[1, col_idx])
    pivot = crop_data.groupby(['label', 'growth_stage'])[col].mean().unstack()
    pivot.columns = [gs_map[c] for c in pivot.columns]
    cmap  = 'Blues' if col == 'rainfall' else 'YlGn'
    im    = ax2.imshow(pivot.values, cmap=cmap, aspect='auto')
    ax2.set_xticks(range(pivot.shape[1])); ax2.set_xticklabels(pivot.columns, fontsize=9)
    ax2.set_yticks(range(pivot.shape[0])); ax2.set_yticklabels(pivot.index, fontsize=9)
    for r in range(pivot.shape[0]):
        for c2 in range(pivot.shape[1]):
            ax2.text(c2, r, f'{pivot.values[r, c2]:.1f}',
                     ha='center', va='center', color='#111',
                     fontsize=9, fontweight='bold')
    cb = plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
    cb.ax.tick_params(colors=WHITE, labelsize=7)
    style_ax(ax2, f'Mean {col}  (crop × stage, soil avg)')

# ── Panel [1,2]: Scatter — rainfall vs soil_moisture coloured by crop ──────────
fig.get_axes()[-1].remove()   # drop the last heatmap (soil_moisture col 2 row 1)
ax_s = fig.add_subplot(gs_fig[1, 2])
for crop in CROPS:
    sub = crop_data[crop_data['label'] == crop]
    ax_s.scatter(sub['rainfall'], sub['soil_moisture'],
                 color=CROP_COLORS[crop], alpha=0.45, s=20, label=crop)
ax_s.legend(fontsize=8, labelcolor=WHITE, facecolor=AX_BG, edgecolor=GREY)
style_ax(ax_s, 'Rainfall vs Soil Moisture by Crop',
         xlabel='Rainfall (mm)', ylabel='Soil Moisture (%)')

# plt.savefig('water_rulebook_viz.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# --- Lookup helpers ---

def get_range(rulebook, *keys, column):
    """Return {min, max, mean, std} for a given key path + column, or None if missing."""
    try:
        node = rulebook
        for k in keys:
            node = node[k]
        return node[column]
    except KeyError:
        return None


def in_range(rulebook, *keys, column, value):
    """Return True/False if value falls within [min, max], or None if no rule exists."""
    rule = get_range(rulebook, *keys, column=column)
    if rule is None:
        return None
    return rule['min'] <= value <= rule['max']


# Demo — each rulebook with its own key signature
print(in_range(climate_rulebook,        'rice', 1, 2, column='temperature',  value=25.0))
print(in_range(soil_chemistry_rulebook, 'rice',        column='ph',           value=7.0))
print(in_range(water_rulebook,          'rice', 1, 2, column='rainfall',      value=210.0))

## Irrigation Frequency Rulebook

`irrigation_frequency` is an ordinal integer (1–6). Before choosing the grouping key, we compare four candidate levels on two axes:
- **Between-group spread** — do groups actually differ? (higher = more signal)
- **Within-group std** — how wide is the spread inside each group? (lower = tighter rules)
- **Min group size** — is there enough data per group to trust the stats?

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# IRRIGATION SCHEDULE — Key Selection Analysis
#
# irrigation_frequency is ordinal (1–6 times/week).
# Goal: one recommended value per group, not a range.
# Metric: round(mean) — more stable than mode in groups of 6–15 rows.
#
# We compare three candidate keys and check whether water_source adds value.
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
from scipy import stats as sp_stats

TARGET   = 'irrigation_frequency'
gs_map   = {1: 'Seedling', 2: 'Growing', 3: 'Mature'}
st_map   = {1: 'Soil 1',   2: 'Soil 2',  3: 'Soil 3'}

LEVELS = [
    ('crop + stage',         ['label', 'growth_stage']),
    ('crop + stage + soil',  ['label', 'growth_stage', 'soil_type']),
    ('+ water source',       ['label', 'growth_stage', 'soil_type', 'water_source_type']),
]

# ── Compute per-level diagnostics ─────────────────────────────────────────────
level_data = {}
for name, keys in LEVELS:
    rows = []
    for group_keys, grp in crop_data.groupby(keys)[TARGET]:
        gk = (group_keys,) if not isinstance(group_keys, tuple) else group_keys
        rec = round(grp.mean())
        rows.append({
            'keys'     : gk,
            'n'        : len(grp),
            'mean'     : round(grp.mean(), 2),
            'mode'     : int(sp_stats.mode(grp, keepdims=True).mode[0]),
            'rec'      : int(rec),
            'reliable' : len(grp) >= 6,
        })
    level_data[name] = rows

# ─── Print summary ────────────────────────────────────────────────────────────
for name, rows in level_data.items():
    n_total    = len(rows)
    n_reliable = sum(r['reliable'] for r in rows)
    min_n      = min(r['n'] for r in rows)
    print(f'{name:30s}  groups={n_total:3d}  reliable={n_reliable}/{n_total}  min_n={min_n}')

# ─────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor(BG)
gs_fig = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.40)
fig.suptitle(
    'Irrigation Schedule — Key Selection Analysis\n'
    '✓ Chosen: crop + growth_stage + soil_type  |  Metric: round(mean)',
    color=WHITE, fontsize=12, fontweight='bold', y=0.99
)

# ── Panel 1: Min group size per level (reliability bar chart) ─────────────────
ax1 = fig.add_subplot(gs_fig[0, 0])
level_names = [n for n, _ in LEVELS]
min_sizes   = [min(r['n'] for r in level_data[n]) for n in level_names]
colors_bar  = [GREEN if ms >= 6 else RED for ms in min_sizes]
short_names = ['crop+stage', '+soil', '+water_src']
bars = ax1.bar(short_names, min_sizes, color=colors_bar, alpha=0.85)
for bar, val in zip(bars, min_sizes):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(val), ha='center', color=WHITE, fontsize=9)
ax1.axhline(6, color=AMBER, linewidth=1.5, linestyle='--', alpha=0.8)
ax1.text(2.45, 6.1, 'min=6', color=AMBER, fontsize=8, ha='right')
style_ax(ax1, 'Min Group Size per Level', ylabel='Min rows in any group')

# ── Panel 2: Mode vs round(mean) stability at crop+stage ─────────────────────
ax2 = fig.add_subplot(gs_fig[0, 1])
rows_cs = level_data['crop + stage']
x_pos   = range(len(rows_cs))
modes   = [r['mode'] for r in rows_cs]
recs    = [r['rec']  for r in rows_cs]
x_lbls  = [f"{r['keys'][0][:3]}\n{gs_map[r['keys'][1]][:4]}" for r in rows_cs]
w = 0.35
ax2.bar([i - w/2 for i in x_pos], modes, width=w, color=PURPLE, alpha=0.8, label='Mode')
ax2.bar([i + w/2 for i in x_pos], recs,  width=w, color=GREEN,  alpha=0.8, label='round(mean)')
ax2.set_xticks(list(x_pos)); ax2.set_xticklabels(x_lbls, fontsize=7.5)
ax2.set_ylim(0, 7)
ax2.legend(fontsize=8, labelcolor=WHITE, facecolor=AX_BG, edgecolor=GREY)
style_ax(ax2, 'Mode vs round(mean)  [crop + stage]\nmode is noisier — mean is more stable',
         ylabel='Irrigation frequency')

# ── Panel 3: How much does soil_type shift the recommendation? ────────────────
ax3 = fig.add_subplot(gs_fig[0, 2])
# For each (crop, stage), show spread of recommendations across soil types
spread_data = {}
rows_css = level_data['crop + stage + soil']
for r in rows_css:
    crop, stage, soil = r['keys']
    key = f"{crop[:3]}\n{gs_map[stage][:4]}"
    spread_data.setdefault(key, []).append(r['rec'])

keys_sorted = list(spread_data.keys())
spreads     = [max(v) - min(v) for v in spread_data.values()]
bar_cols    = [AMBER if s >= 2 else BLUE for s in spreads]
bars3 = ax3.bar(range(len(keys_sorted)), spreads, color=bar_cols, alpha=0.85)
ax3.set_xticks(range(len(keys_sorted)))
ax3.set_xticklabels(keys_sorted, fontsize=7)
ax3.axhline(2, color=AMBER, linewidth=1.2, linestyle='--', alpha=0.7)
ax3.text(len(keys_sorted) - 0.55, 2.1, '≥2 shift', color=AMBER, fontsize=7.5)
patches_leg = [mpatches.Patch(color=AMBER, label='soil shifts rec ≥2'),
               mpatches.Patch(color=BLUE,  label='soil shift <2')]
ax3.legend(handles=patches_leg, fontsize=7, labelcolor=WHITE, facecolor=AX_BG, edgecolor=GREY)
style_ax(ax3,
         'Soil-type effect on recommendation\n(max−min across soils per crop+stage)',
         ylabel='Rec. range across soil types')

# ── Panels 4–6: Recommendation heatmaps — one per soil type ──────────────────
CHOSEN_LEVEL = 'crop + stage + soil'
soil_types   = sorted(crop_data['soil_type'].unique())

for si, soil in enumerate(soil_types):
    ax = fig.add_subplot(gs_fig[1, si])
    soil_rows = [r for r in level_data[CHOSEN_LEVEL] if r['keys'][2] == soil]
    crops_u  = sorted(set(r['keys'][0] for r in soil_rows))
    stages_u = sorted(set(r['keys'][1] for r in soil_rows))

    matrix = np.zeros((len(crops_u), len(stages_u)), dtype=int)
    for r in soil_rows:
        ri = crops_u.index(r['keys'][0])
        ci = stages_u.index(r['keys'][1])
        matrix[ri, ci] = r['rec']

    im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto', vmin=1, vmax=6)
    ax.set_xticks(range(len(stages_u)))
    ax.set_xticklabels([gs_map[s] for s in stages_u], fontsize=9)
    ax.set_yticks(range(len(crops_u))); ax.set_yticklabels(crops_u, fontsize=9)
    for r in range(matrix.shape[0]):
        for c in range(matrix.shape[1]):
            ax.text(c, r, str(matrix[r, c]), ha='center', va='center',
                    color='#111', fontsize=13, fontweight='bold')
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(colors=WHITE, labelsize=7)
    style_ax(ax, f'{st_map[soil]} — Recommended irrigations/day\n(round of mean)')

# plt.savefig('irrigation_schedule_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ── Build irrigation_schedule  (key: crop + growth_stage + soil_type) ─────────
# Stores one recommended frequency per combination, not a range.
# recommended_per_week = round(mean) — more reliable than mode at n=6–15.

IRRIG_KEY_COLS = ['label', 'growth_stage', 'soil_type']

irrigation_schedule = {}
for (crop, stage, soil), grp in crop_data.groupby(IRRIG_KEY_COLS)['irrigation_frequency']:
    irrigation_schedule \
        .setdefault(crop, {}) \
        .setdefault(int(stage), {}) \
        [int(soil)] = {
            'recommended_per_week': int(round(grp.mean())),
            'mean'               : round(float(grp.mean()), 3),
            'n'                  : int(len(grp)),
        }

# ── Schedule lookup helper ────────────────────────────────────────────────────
def get_irrigation(crop, growth_stage, soil_type):
    """Return recommended daily irrigation frequency, or None if not found."""
    try:
        return irrigation_schedule[crop][int(growth_stage)][int(soil_type)]['recommended_per_week']
    except KeyError:
        return None

# Flat DataFrame for export
irrig_records = []
for crop, stages in irrigation_schedule.items():
    for stage, soils in stages.items():
        for soil, entry in soils.items():
            irrig_records.append({'crop': crop, 'growth_stage': stage,
                                  'soil_type': soil, **entry})
irrigation_df = pd.DataFrame(irrig_records)
irrigation_df